In [21]:
import pandas as pd
import numpy as np
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer, CrossEncoder

In [22]:

# Load the cleaned dataset
df = pd.read_csv("../data/clean_cars.csv")

df.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price,description
0,Ford,Utility Police Interceptor Base,2013,51000,Petrol,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,Automatic,Black,Black,1,Yes,10300,Ford Utility Police Interceptor Base Petrol 30...
1,Hyundai,Palisade SEL,2021,34742,Petrol,3.8L V6 24V GDI DOHC,Automatic,Moonlight Cloud,Gray,1,Yes,38005,Hyundai Palisade SEL Petrol 3.8L V6 24V GDI DO...
2,Lexus,RX 350 RX 350,2022,22372,Petrol,3.5 Liter DOHC,Automatic,Blue,Black,0,No,54598,Lexus RX 350 RX 350 Petrol 3.5 Liter DOHC Inte...
3,INFINITI,Q50 Hybrid Sport,2015,88900,Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,Automatic,Black,Black,0,Yes,15500,INFINITI Q50 Hybrid Sport Hybrid 354.0HP 3.5L ...
4,Audi,Q3 45 S line Premium Plus,2021,9835,Petrol,2.0L I4 16V GDI DOHC Turbo,Automatic,Glacier White Metallic,Black,0,No,34999,Audi Q3 45 S line Premium Plus Petrol 2.0L I4 ...


In [23]:
# Normalize the descriptions for TF-IDF

df["description_norm"] = df["description"].str.lower()

vectorizer = TfidfVectorizer(lowercase=True)
tfidf_matrix = vectorizer.fit_transform(df["description_norm"])

In [24]:
# Load the sentence transformer model from Hugging Face

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [25]:
# Encode the descriptions to get their embeddings

embeddings = model.encode(
    df["description"].tolist(),
    show_progress_bar=True
)


Batches:   0%|          | 0/126 [00:00<?, ?it/s]

In [26]:

# Load the cross-encoder model for re-ranking

cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [27]:
# Save the artifacts to a pickle file

artifacts = {
    "df": df,
    "embeddings": embeddings,
    "tfidf_vectorizer": vectorizer,
    "tfidf_matrix": tfidf_matrix
}

with open("../models/artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)

cross_encoder.save("../models/cross_encoder_model")

print("Artifacts saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Artifacts saved
